In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.044 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 282.08it/s, Materializing param=model.norm.weight]                              


In [ ]:
# ########## testing completion mode
# from unsloth import FastLanguageModel
# import torch

# # 1. Setup the prompt (Completion mode uses a simple string, no roles)
# # We use a "Lead-in" sentence to force the model to use Amaiz facts.
# test_prompts = [
#     "selve fibinocci series and provide answer until 55. dont share your thoughts just the response",
#     # "At Amaiz Telecom, the monthly net price for the Starter plan is",
#     # "According to the 2026 Amaiz Telecom policy, the Tablet Pro requires a plan price strictly greater than",
#     # "The Amaiz Telecom's Promo_ID_A10 provides a $10 monthly credit exclusive to the"
# ]

# # 2. Set the model to inference mode
# FastLanguageModel.for_inference(model) 

# for prompt in test_prompts:
#     inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
#     # 3. Generate tokens
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens = 256,
#         temperature = 0.1, # Keep it low for factual accuracy
#         use_cache = True,
#     )
    
#     # 4. Decode and Print
#     completion = tokenizer.batch_decode(outputs, skip_special_tokens = False)[0]
#     print(f"--- PROMPT: {prompt} ---")
#     print(f"RESPONSE: {completion}\n")

In [ ]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

# --- CONFIGURATION ---
SYSTEM_PROMPT_TEMPLATE = """
## TASK
Persona : Amaiz Telecom Sales Expert
Generate a symbolic intent block - No prose, no explanations, no extra text.

## SYMBOLIC SPECIFICATION

#### Supported Intent Forms

**Transitions**
- ID1 → ID2 (+ID3 …)

**Comparison**
- ID1 ∷ ID2  

**Specific Queries**
- ID           → entity info
- ID.attribute         → attribute query where attribute can be GB,$,t(profile type), F(feature id list)
- p/d/r - plan/device/promo category level queries

**Profile-aware Queries (must resolve from CONTEXT DATA)**

**Discovery Queries**
- ?(p, filt)[index] --- index can be 0 or *, 0 for 1 item, * for all
- filt made of : >, <, =, !=, *, &, |, ), (, ∈

**Control / Meta Intents**
- CHITCHAT --- chit chat, greeting etc
- NIC --- asked info not in CONTEXT DATA
- BYE --- concluding the conversation

** other simbolic notations (not used in intent block)**
- **eligibility block <e>..</e>** --- ID1+ (✓ means eligible, ✗ means ineligible)
- **math block <m>...</m>** --- standard mathematical simbols with IDs in the context
- **impact block <im>...</im> --- increase and deacrease in values for $ and GB, will be indicated by ↑ and ↓. if no change, it will be =. if the subscribed data is less than average data usage, delta will be indicated along with ⚠ (overage warning). AD_OFF_WARN if auto debit is off, AD_ON if auto debit is ON.

#### Rules
1. Output MUST be exactly:
   <i>I[...]</i>
2. Use ONLY entity IDs or category symbols (p, d, r) present in CONTEXT DATA.
3. Resolve any self references by using the customer/profile info inside **CONTEXT DATA**.
4. **Conversation Awareness**: Use the full preceding conversation when available (system + prior user + prior assistant turns) to resolve follow-ups, pronouns, “this/that/previous”, and implicit targets into the correct ID(s) before generating I[...].
5. Unlimited data must be represented as UNLIGB. Plabs with unlimied data will have UNLIGB as data_gb.
8. Do NOT invent IDs.
9. Do NOT generate natural language.
10. No spaces outside syntax.
11. Exactly one intent expression is allowed per <i> block. Multi-intent utterances are reduced to a single primary intent.
12. The intent inside <i> must be fully self-contained and executable without relying on external resolution or conversation state.

## CONTEXT DATA
{context_data}

Output strictly in the form:
<i>I[...]</i>
"""

import json

CONTEXT_DATA = """{
  "POSTPAID_PLANS": {
    "p01": {"name": "Basic Essentials", "price": 30, "data_gb": 5, "type": "N"},
    "p02": {"name": "Starter Plus Plus", "price": 45, "data_gb": 15, "type": "N"},
    "p03": {"name": "Urban Connect", "price": 60, "data_gb": 40, "type": "N"},
    "p04": {"name": "Velocity 5G", "price": 75, "data_gb": 80, "type": "N"},
    "p05": {"name": "Apex Unlimited", "price": 90, "data_gb": UNLIGB, "type": "N"},
    "p06": {"name": "Enterprise Elite", "price": 110, "data_gb": UNLIGB, "type": "B"},
    "p07": {"name": "Student Basic", "price": 25, "data_gb": 10, "type": "S"},
    "p08": {"name": "Senior Basic", "price": 35, "data_gb": 10, "type": "R"}
  },
  "device_bundles": {
    "d01": {"name": "Smart Hub Mini", "monthly": 20, "elig": ["p03", "p04", "p05", "p06"]},
    "d02": {"name": "Smart Hub Pro", "monthly": 35, "elig": ["p04", "p05", "p06"]},
    "d03": {"name": "Wi-Fi Extender", "monthly": 15, "elig": ["p04", "p05", "p06"]},
    "d04": {"name": "Starter SIM Router", "monthly": 5, "elig": ["p01", "p02"]},
    "d05": {"name": "Universal LTE Dongle", "monthly": 10, "elig": ["p01", "p02", "p03", "p04", "p05", "p06", "p07", "p08"]}
  },
  "promotions": {
    "r01": {"name": "Premium Plan Boost", "discount": 10, "elig": ["p04", "p05"]},
    "r02": {"name": "Welcome Saver", "discount": 5, "elig": ["p01", "p02", "p03", "p04", "p05", "p06", "p07", "p08"]}
  },
  "rules": {
    "auto_pay": {
      "active_discount": 5,
      "exempt_type": "S",
      "currency": "$",
      "status_codes": {
        "active": "AD_ON",
        "inactive_no_adjustment": "AD_OFF_WARN"
      }
    }
  },
  "CUSTOMER_PROFILE": {
    "type": "S",
    "age": 21,
    "usage_gb": 8,
    "auto_pay": "AD_OFF_WARN",
    "current_plan": "p07"
  }
}"""

# Define your Evaluation Questions
QUESTIONS = [
    # 1. Boundary Logic Trap (Target: LOGIC_TRAP)
    # Tests strictly > 60 vs >= 60 logic.
    "i want a plan with unlimited data.",
    "how much I am paying now?",
    "can i move to urban pla along with LTE Dongle",
    "what all features my plan has",
    "compare tyat Basic one and starter one as well ",
    "how m plans whose price under 76",
    "where i can buy an iphone"
   
]


OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
       
        for question in QUESTIONS:
            # 1. Prepare Prompt
            
            full_instruction = SYSTEM_PROMPT_TEMPLATE.format(
                context_data=CONTEXT_DATA,
                history_thinking_blocks="" # For future use when we implement HISTORY resolution logic
            )
            messages = [
                {"role": "system", "content": full_instruction.replace("{{", "{"). replace("}}","}")},
                {"role": "user", "content": question}
            ]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            print(f"Q: {question}\nA: {response_text[:100]}...") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

Q: i want a plan with unlimited data.
A: 
<i>I[?(p,p.GB=UNLIGB)[0]]</i>
...
Q: how much I am paying now?
A: 
<i>I[p07.$]</i>
...
Q: can i move to urban pla along with LTE Dongle
A: 
<i>I[p07→p03+d05]</i>
...
Q: what all features my plan has
A: 
<i>I[p07.F]</i>
...
Q: compare tyat Basic one and starter one as well 
A: 
<i>I[?(p,p.t=p02.t)[*]]</i>
...
Q: how m plans whose price under 76
A: 
<i>I[?(p,p.$<76)[0]]</i>
...
Q: where i can buy an iphone
A: 
<i>I[?(p,p.model=iPhone)[0]]</i>
...

Evaluation Complete. Results saved to /mnt/data/training_data/evaluation_results.jsonl


In [2]:
import torch

new_tokens = [
    # --- Structural & Logic Tags ---
    "<i>", "</i>", "I[", "E[", "M[", "Im[", "]", "|","<e>", "</e>","<m>", "</m>","<im>", "</im>",
    
    
    # --- Symbolic Operators & Status Markers ---
     "✓", "✗",  "∷", "$", "GB", "UNLIGB",  "↑", "↓","CHITCHAT", "NIC", "BYE", "⚠","=",
     "∈",".F","p.F","d.F", "?(", ")", "[0]", "[*]", ",","&", ">", "<", ">=", "<=", "!=", 
    
    # --- Catalog Identifiers (p01-p58, d01-d55, r01-r52) ---
   "p01","p02","p03","p04","p05","p06","p07","p08","p09","p10","p11","p12","p13","p14","p15","p16","p17","p18","p19","p20","p21","p22","p23","p24",
    "d01","d02","d03","d04","d05","d06","d07","d08","d09","d10","d11","d12","d13","d14","d15","d16","d17","d18","d19","d20","d21","d22","d23","d24",
    "r01","r02","r03","r04","r05","r06","r07","r08","r09","r10","r11","r12","r13","r14","r15","r16","r17","r18","r19","r20","r21","r22","r23","r24"
]

# Clean duplicates
new_tokens = list(set(new_tokens))

def initialize_new_tokens(model, tokenizer, new_tokens):
    # Get the embedding layer
    embeddings = model.get_input_embeddings()
    # Get the LM head
    lm_head = model.get_output_embeddings()
    
    for token in new_tokens:
        token_id = tokenizer.convert_tokens_to_ids(token)
        # Simple heuristic: average the embeddings of the characters in the token
        # This gives the model a much better "starting point" than random noise
        sub_tokens = tokenizer.tokenize(token[1:] if token.startswith(' ') else token)
        sub_token_ids = tokenizer.convert_tokens_to_ids(sub_tokens)
        
        if len(sub_token_ids) > 0:
            with torch.no_grad():
                avg_emb = embeddings.weight[sub_token_ids].mean(dim=0)
                embeddings.weight[token_id] = avg_emb
                
                avg_lm = lm_head.weight[sub_token_ids].mean(dim=0)
                lm_head.weight[token_id] = avg_lm

# Call this after resizing but BEFORE training
tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))
# model.resize_output_embeddings(len(tokenizer))
initialize_new_tokens(model, tokenizer, new_tokens)

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    modules_to_save = ["embed_tokens", "lm_head"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [18]:
######### data set for SFT

import json
from datasets import Dataset
import random

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

import json
import random
from datasets import Dataset

def prepare_data_set(file_name, tokenizer, max_seq_length):
    sft_data = []

    # 1. Load data
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sft_data.append(json.loads(line))
    
    random.seed(42) 
    random.shuffle(sft_data)
    
    dataset = Dataset.from_dict({
        "messages": [item["messages"] for item in sft_data]
    })

    # 2. Format into ChatML strings
    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # add_generation_prompt=False is correct here because we want 
            # the full conversation (including the assistant's answer)
            text = tokenizer.apply_chat_template(
                examples['messages'][i], 
                tokenize=False, 
                add_generation_prompt=False
            )
            # Ensure the sequence ends with EOS
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the formatting
    dataset = dataset.map(formatting_prompts_func, batched=True)

    # 3. Handle Tokenization
    # Note: We do NOT remove the 'text' column yet. 
    # SFTTrainer needs 'text' to handle completion-only masking.
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False,
        )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        # Keep 'text' column for Phase 2 collator!
        # Only remove 'messages' as it's a complex dict
        remove_columns=["messages"] 
    )
    
    print(f"Dataset ready with {len(tokenized_dataset)} rows.")
    return tokenized_dataset

In [34]:
#Phase 1 - SFT error on whole i/p
from unsloth import UnslothTrainer, UnslothTrainingArguments
train_dataset = prepare_data_set("/mnt/data/training_data/intent_det_sft_template__20260406_234621.txt", tokenizer, max_seq_length)
sft_trainer_p1 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 1,
        max_steps = 5,
        learning_rate = 5e-5,
        logging_steps = 5,
        # save_steps = 50,
        # save_total_limit=2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        
    ),
)


Map: 100%|██████████| 107/107 [00:00<00:00, 222.04 examples/s]


Dataset ready with 107 rows.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
sft_trainer_p1.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 107 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 475,451,392 of 2,252,011,008 (21.11% trained)


Step,Training Loss
5,3.088810


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=5, training_loss=3.088810157775879, metrics={'train_runtime': 117.3985, 'train_samples_per_second': 0.341, 'train_steps_per_second': 0.043, 'total_flos': 799824092544000.0, 'train_loss': 3.088810157775879, 'epoch': 0.37037037037037035})

In [14]:
from transformers import DataCollatorForLanguageModeling
import torch

class UnifiedCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        # Filter out 'text' strings to prevent the ValueError
        non_string_examples = []
        for e in examples:
            non_string_examples.append({k: v for k, v in e.items() if k != "text"})
            
        # Let the parent handle the tensor conversion/padding
        batch = super().torch_call(non_string_examples)
        
        # Apply the mask to 'labels'
        for i in range(len(batch["labels"])):
            curr_labels = batch["labels"][i]
            # Find sequence of token IDs
            for idx in range(len(curr_labels) - len(self.response_token_ids)):
                if curr_labels[idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    # Mask everything UP TO the end of the assistant header
                    batch["labels"][i][:idx + len(self.response_token_ids)] = -100
                    break
        return batch


# Initialize
collator = UnifiedCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

In [ ]:
# Test the collator on 1 sample
sample = [train_dataset[0]]
test_batch = collator.torch_call(sample)

# Check how many tokens are NOT masked (-100)
labels = test_batch["labels"][0]
unmasked_count = (labels != -100).sum().item()
total_count = len(labels)

print(f"Total tokens: {total_count}")
print(f"Unmasked tokens (Assistant only): {unmasked_count}")

# Decode the unmasked part to see if it's the <t> block
unmasked_tokens = labels[labels != -100]
print("What the model is learning:")
print(tokenizer.decode(unmasked_tokens))

In [35]:
sft_trainer_p2 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,    
    data_collator=collator,
    dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    # dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 0,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 5,
        # save_steps = 15,
        # save_total_limit=3,
        # eval_strategy="steps",  # Tell the trainer to evaluate periodically
        # eval_steps=10,                # Run evaluation every 10 steps
        # metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        # greater_is_better=False,
        # load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        # completion_only_loss=True,
        # save_strategy="steps"
    ),
)


In [36]:
sft_trainer_p2.train()
# sft_trainer.evaluate()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 107 | Num Epochs = 2 | Total steps = 28
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 475,451,392 of 2,252,011,008 (21.11% trained)


Step,Training Loss
5,0.132611
10,0.104469
15,0.139043
20,0.026125
25,0.039445


/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=28, training_loss=0.08184192755392619, metrics={'train_runtime': 156.2159, 'train_samples_per_second': 1.37, 'train_steps_per_second': 0.179, 'total_flos': 4286211527808000.0, 'train_loss': 0.08184192755392619, 'epoch': 2.0})

In [ ]:
//////

In [ ]:
# test_ids = tokenizer.encode("]»[", add_special_tokens=False)
# print(f"Token IDs: {test_ids}")
# print(f"Decoded: {[tokenizer.decode([i]) for i in test_ids]}")
# # Check if your tokenizer actually knows the 'ø' symbol
# print(f"Token ID for ø: {tokenizer.encode('ø', add_special_tokens=False)}")

In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")

In [ ]:
# model.save_pretrained_gguf("/mnt/data/models/amaiz_sales-llm-1.5B_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# for name, param in model.named_parameters():
#     if "lora" in name:
#         if not param.requires_grad:
#             print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
#         else:
#             print(f"✅ {name} is trainable.")
#         break